In [1]:
from bs4 import BeautifulSoup
from datetime import datetime
from pathlib import Path
import json
import re

from lib.helpers.findIssnByType import find_issn_by_type
from lib.helpers.firstStr import first_str

In [2]:
with open("data/html/cv.html", "r", encoding="utf-8") as f:
    html_content = f.read()

In [3]:
soup = BeautifulSoup(html_content, "html.parser")

# Profile

In [21]:
rotulo = soup.find("b", string=lambda s: s and "orcid" in s.lower())
label_cell = rotulo.find_parent("div", class_=lambda c: c and "layout-cell" in c)
value_cell = label_cell.find_next_sibling("div")
value_cell

In [36]:
parent = rotulo.find_parent().find_parent()
sibling = parent.find_next_sibling("div")
sibling.find_all('a')[1].text.strip()

'https://orcid.org/0000-0002-3823-3868'

In [14]:
def get_profile(soup: BeautifulSoup, lattes_id: str) -> dict:
    profile = {'is_inpa_researcher': True}
    infpessoa = soup.find('div', class_='infpessoa')
    full_name = infpessoa.find('h2').text.strip()
    given_name = full_name.split()[0]
    family_name = " ".join(full_name.split()[1:]) if len(full_name.split()) > 1 else None
    profile["full_name"] = full_name
    profile["given_name"] = given_name
    profile["family_name"] = family_name
    profile["lattes_id"] = lattes_id
    # Extract ORCID
    orcid_label = soup.find("b", string=lambda s: s and "orcid" in s.lower())
    if orcid_label:
        parent = orcid_label.find_parent().find_parent()
        sibling = parent.find_next_sibling("div")
        if sibling:
            orcid = sibling.find_all('a')[1].text.strip()
            orcid = orcid.replace("https://orcid.org/", "")
            profile["orcid"] = orcid
            
    cv = { "author": profile}

    return cv

cv = get_profile(soup, '2747150211073176')

In [15]:
cv

{'author': {'is_inpa_researcher': True,
  'full_name': 'Adalberto Luis Val',
  'given_name': 'Adalberto',
  'family_name': 'Luis Val',
  'lattes_id': '2747150211073176',
  'orcid': '0000-0002-3823-3868'}}

# Lattes

In [19]:
def get_lattes_update(soup, cv):
    lattes = { 'lattes_id': cv['author']['lattes_id']}
    
    infoautor = soup.find("ul", class_="informacoes-autor")
    infoupdate = infoautor.find_all("li")[-1].text
    match = re.search(r"(\d{2}/\d{2}/\d{4})", infoupdate)
    if match:
        lattes['lattes_update'] = datetime.strptime(match.group(1), "%d/%m/%Y").date()
    cv['lattes'] = lattes
    return cv

cv = get_lattes_update(soup, cv)
cv

{'author': {'is_inpa_researcher': True,
  'full_name': 'Adalberto Luis Val',
  'given_name': 'Adalberto',
  'family_name': 'Luis Val',
  'lattes_id': '2747150211073176',
  'orcid': '0000-0002-3823-3868'},
 'lattes': {'lattes_id': '2747150211073176',
  'lattes_update': datetime.date(2026, 3, 31)}}

In [20]:
with open("data/json/2747150211073176.json", "w", encoding="utf-8") as f:
    json.dump(cv, f, default=str, indent=4)

# Artigos completos

In [ ]:
artigos = []
with open('data/artigos/val.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        artigos.append(json.loads(line))
artigo = artigos[0]

In [27]:
def parser_container(data):
    container = {
        "name": first_str(data.get("container-title")),
        "alternate_name": first_str(data.get("short-container-title")),
        "publisher": data.get("publisher"),
        "issn_print": find_issn_by_type(data.get("issn-type"), "print"),
        "issn_electronic": find_issn_by_type(data.get("issn-type"), "electronic"),
        "isbn": first_str(data.get("ISBN"))
        }
    return container
container = parser_container(artigo)